In [32]:
# initialize model

# Per Round:
# server copies and sends the global model weights.
# client trains locally (on their local dataset).
# client returns its new weights and sample count.
# server aggregates the weights using FedAvg.
# global model is updated.
# (Optional) Evaluate global model on a held-out test set.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import json
import shutil

In [34]:
from federated_multihead_model import SharedEncoders, TabularClientModel, ImageClientModel, MultiClientModel
from config import D_TABULAR, D_EMBEDDING, D_FUSION

from tabular_site_training import training as complete_tabular_training
'''
from image_site_training   import training_loop as image_training_loop
from multi_site_training   import training_loop as multi_training_loop
'''

'\nfrom image_site_training   import training_loop as image_training_loop\nfrom multi_site_training   import training_loop as multi_training_loop\n'

In [35]:
# reimport all files
import importlib
import federated_multihead_model
import tabular_site_training
importlib.reload(federated_multihead_model)
importlib.reload(tabular_site_training)

<module 'tabular_site_training' from 'd:\\USYD\\2025 S2\\5703 Capstone\\model\\tabular_site_training.py'>

In [36]:
# 1. build local model: global encoders + local heads:
def build_local_model (name, modality, n_classes, save_path):
    # 1. define the complete model structure
    # solved with import
    
    # 2. assemble local model
    # 2.1 Initialize local encoders
    encoder_placeholder = SharedEncoders(
        d_tabular   = D_TABULAR, 
        d_embedding = D_EMBEDDING, 
        d_fusion    = D_FUSION
        )
    # 2.2 Load the weights
    # no need here

    # 2.3 build local model
    # modality check
    if modality == "tabular":
        local_model = TabularClientModel(shared_encoders = encoder_placeholder, n_classes = n_classes)
    elif modality == "image":
        local_model = ImageClientModel  (shared_encoders = encoder_placeholder, n_classes = n_classes)
    elif modality == "multi":
        local_model = MultiClientModel  (shared_encoders = encoder_placeholder, n_classes = n_classes)
    else:
        # report ERROR
        exit()
    
    # 2.4 save the local head weights
    torch.save(local_model.head.state_dict(), save_path)
    return save_path

In [37]:
def assign_local_training_function (modality):
    if modality == "tabular":
        return complete_tabular_training
    elif modality == "image":
        pass
    elif modality == "multi":
        pass
    else:
        # report ERROR
        return None

In [38]:
def selective_fedavg(client_states, client_weights):
    """FedAvg per key (only keys that were returned by clients)"""
    from collections import defaultdict

    grouped = defaultdict(list)

    for state, weight in zip(client_states, client_weights):
        for k, v in state.items():
            grouped[k].append((v, weight))

    avg_state = {}
    for k, updates in grouped.items():
        total_weight = sum(w for _, w in updates)
        avg_state[k] = sum((v * (w / total_weight)) for v, w in updates)

    return avg_state

In [39]:
def local_training (global_state, site):
    if site["modality"] == "tabular":
        dataset_path    = site["clean_dataset_path"]
        label_col_name  = site["label_col"]
        num_of_classes  = site["n_classes"]
        local_head_path = site["head_ckpt_path"]
        return site["site_training"](global_state, dataset_path, label_col_name, num_of_classes, local_head_path)
    
    elif site["modality"] == "image":
        dataset_path    = site["clean_dataset_path"]
        num_of_classes  = site["n_classes"]
        local_head_path = site["head_ckpt_path"]
        return site["site_training"](global_state, dataset_path, num_of_classes, local_head_path)

    elif site["modality"] == "multi":
        return None
    
    else:
        print("site modality error")
        exit()

In [ ]:
def federated_training_one_round (global_state, sites):
    # 1. sever send global encoder weights to sites
    # by passing global_state
    
    ################ COME TO EACH SITES ##################
    client_updates = []
    for site in sites:
        # 2. site preprocessing
        # return: "clean_dataset_path": str, "label_col": str, "n_classes": int

        # BEST SOLUTION: Find a way to record the preprocessing pipeline in site_info.json
        pass
        
        # placeholders:
        # site tabular_1
        site["clean_dataset_path"] = "tabular_dataset/diabetes_012_ready_to_model.csv"
        site["label_col"] = "Diabetes_012"
        site["n_classes"] = 2

        # 3. build site head if not exist:
        os.makedirs("local_heads", exist_ok = True)
        head_path = os.path.join("local_heads", f"{site["name"]}.pth")
        # existing: pass
        if os.path.exists(head_path):
            site["head_ckpt_path"] = head_path
        else:
            # NOT existing: initialize
            site["head_ckpt_path"] = build_local_model(site["name"], site["modality"], site["n_classes"], head_path)
        
        # 4. assign the correct local training function
        site["site_training"] = assign_local_training_function(site["modality"])
        # assembling the complete model is during training

        # 5. full local training
        # using all site info and correct modality training function
        updated_state, sample_count = local_training(global_state, site) 
        # record the reaults
        client_updates.append((updated_state, sample_count))
    ################## END in sites, back to server ##################

    # 5. Server aggregates encoder weights using FedAvg (per key, selectively)
    agg_state = selective_fedavg(
        client_states  = [s for s, _ in client_updates],
        client_weights = [w for _, w in client_updates]
    )

    # 6. Server updates global encoders with aggregated weights
    print("Finsh 1 training round")
    return agg_state

In [41]:
# 1. initialize global model
global_encoders = SharedEncoders(
    d_tabular = D_TABULAR, 
    d_embedding = D_EMBEDDING, 
    d_fusion = D_FUSION
    )
global_state = global_encoders.state_dict()

In [42]:
# 2. run build site info (list of dict)

In [43]:
'''
check list:
"name": "tabular_1"  : DONE
"modality": "tabular": DONE
"raw_dataset_path"   : DONE 

"clean_dataset_path": "tabular_dataset/diabetes_012_ready_to_model.csv"
"label_col": "Diabetes_012"
"n_classes": 2

"site_training" : 
"head_ckpt_path": 
'''

'\ncheck list:\n"name": "tabular_1"  : DONE\n"modality": "tabular": DONE\n"raw_dataset_path"   : DONE \n\n"clean_dataset_path": "tabular_dataset/diabetes_012_ready_to_model.csv"\n"label_col": "Diabetes_012"\n"n_classes": 2\n\n"site_training" : \n"head_ckpt_path": \n'

In [44]:
# 2. import all sites (basic info)
with open("sites_info_tabular.json", "r") as f:
    sites = json.load(f)

In [45]:
sites

[{'name': 'tabular_1',
  'modality': 'tabular',
  'raw_dataset_path': 'tabular_dataset/diabetes_012_ready_to_model.csv'}]

In [ ]:
# 3. clean the recorded local heads
folder_path = "local_heads"

if os.path.exists(folder_path):
    shutil.rmtree(folder_path)   # deletes the folder and everything inside
    print(f"Deleted folder: {folder_path}")
else:
    print("Folder does not exist.")


Deleted folder: local_heads


In [ ]:
# 4. operate federated training loop
training_round = 5
best_state = global_state

for i in range(training_round):
    # 1 federated training round -> ALL sites FULLY trained ONCE
    new_state = federated_training_one_round(global_state, sites)
    #  Evaluation on val set
    #  And keep the best (also tell all sites to store the corresponding local head)
    '''
    if new_state is better:
        best_state = new_state
    else:
        pass
    '''
    global_state = new_state

 [best updated] 0.8403 -> runs/exp_final/best.pth
 [best updated] 0.8408 -> runs/exp_final/best.pth
Finsh 1 training round
 [best updated] 0.8370 -> runs/exp_final/best.pth
 [best updated] 0.8371 -> runs/exp_final/best.pth
Finsh 1 training round
 [best updated] 0.8385 -> runs/exp_final/best.pth
 [best updated] 0.8392 -> runs/exp_final/best.pth
Finsh 1 training round
 [best updated] 0.8412 -> runs/exp_final/best.pth
 [best updated] 0.8415 -> runs/exp_final/best.pth
Finsh 1 training round
 [best updated] 0.8427 -> runs/exp_final/best.pth
Finsh 1 training round


In [ ]:
# 4. final evaluation